# Analytics — Plagiarism Detection Evaluation

This notebook evaluates the end-to-end pipeline for a single suspicious document against the **PAN 2011** ground-truth XML annotations. The goal is to measure how well the system identifies plagiarised text spans and classifies their type.

---

## Pipeline Overview

The full detection pipeline has six stages:

| Stage | Description |
|-------|-------------|
| **1. Source Retrieval** | Candidate source documents are ranked using four branches: TF-IDF, ESA, LSA, and dense embeddings. The top-20 fused candidates are passed forward. |
| **2. Chunk-Pair Extraction** | Each suspicious chunk is matched against all candidate source chunks using embedding similarity. The top-25 highest-scoring pairs per source document are kept. |
| **3. LLM Source Confirmation** | An LLM scores each source document (0–1) for likelihood of being a true plagiarism source. Documents below a threshold (0.95) are dropped. |
| **4. Plagiarism Type Classification** | An LLM classifies each remaining chunk pair into one of four types: `copy_paste`, `paraphrase`, `shake`, or `none`. |
| **5. Span Merging** | Consecutive chunk-level detections from the same source document are merged into contiguous suspicious spans (gap ≤ 1 800 chars). |
| **6. Ground-Truth Evaluation** | Merged spans are compared against XML ground-truth annotations using character-offset overlap on the suspicious side. |

---

## Evaluation Metrics

Detection quality is measured at **span level** — each merged detected span is compared against each ground-truth span for the same document.

| Metric | Formula | Meaning |
|--------|---------|---------|
| **True Positive (TP)** | Detected span overlaps a GT span (same source doc + suspicious range) | Correctly found plagiarism |
| **False Positive (FP)** | Detected span has no overlapping GT span | Spurious detection |
| **False Negative (FN)** | GT span not covered by any detected span | Missed plagiarism |
| **Precision** | TP / (TP + FP) | Of everything flagged, how much is real |
| **Recall** | TP / (TP + FN) | Of everything real, how much was found |
| **F1** | 2 · P · R / (P + R) | Harmonic mean — balances P and R |

> **Note:** Source-side character offsets are not required to match. Source documents in PAN 2011 are large concatenated files; semantically similar chunks can appear in many different sections of the same source. Suspicious-side overlap is therefore the authoritative criterion.

---

## Plagiarism Type Taxonomy

The classifier labels each chunk pair with one of four types:

| Type | Description |
|------|-------------|
| `copy_paste` | Text is copied verbatim or near-verbatim (< 5 % change) |
| `paraphrase` | Meaning preserved but sentences restructured or rewritten |
| `shake` | Words replaced by synonyms / light edits, same sentence structure |
| `none` | No meaningful plagiarism detected between the two chunks |

---

## Key Parameters (tunable)

| Parameter | Default | Effect |
|-----------|---------|--------|
| `LLM_SCORE_THRESHOLD` | 0.95 | Minimum LLM confidence to keep a source document |
| `TOP_PAIRS_PER_DOC` | 25 | Max chunk pairs sent to the LLM per source document |
| `MAX_GAP` | 1 800 chars | Maximum gap between consecutive chunks to merge into one span |

In [4]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../../../datasets/processed/PAN2011_300")
SUSPICIOUS_DOC_ID = "part1__suspicious-document00007.txt"

# Step 1 - load everything
candidates_df = pd.read_parquet(PROCESSED_DIR / "embedding_candidates_suspicious.parquet")
top20_df = pd.read_parquet(Path("../../top20_df.parquet"))
source_chunks = pd.read_parquet(PROCESSED_DIR / "source_chunks.parquet")
suspicious_chunks = pd.read_parquet(PROCESSED_DIR / "suspicious_chunks_embeddings.parquet")

# Step 2 - filter candidates to top 20 source docs only
top_source_ids = set(top20_df["source_doc_id"].tolist())

candidates_filtered = candidates_df[
    (candidates_df["suspicious_doc_id"] == SUSPICIOUS_DOC_ID) &
    (candidates_df["source_doc_id"].isin(top_source_ids))
].copy()

print(f"Filtered candidates: {len(candidates_filtered)}")
print(candidates_filtered.columns.tolist())

Filtered candidates: 1193
['suspicious_chunk_id', 'suspicious_doc_id', 'suspicious_chunk_index', 'suspicious_start_char', 'suspicious_end_char', 'source_chunk_id', 'source_doc_id', 'source_chunk_index', 'source_start_char', 'source_end_char', 'embedding_score', 'embedding_rank']


In [ ]:
susp_text_col = "embedding_text"
src_text_col  = "chunk_text"

susp_text = (
    suspicious_chunks[suspicious_chunks["doc_id"] == SUSPICIOUS_DOC_ID]
    [["chunk_id", susp_text_col]]
    .rename(columns={"chunk_id": "suspicious_chunk_id", susp_text_col: "suspicious_text"})
)

src_text = (
    source_chunks[source_chunks["doc_id"].isin(top_source_ids)]
    [["chunk_id", src_text_col]]
    .rename(columns={"chunk_id": "source_chunk_id", src_text_col: "source_text"})
)

pairs_df = (
    candidates_filtered
    .merge(susp_text, on="suspicious_chunk_id", how="inner")
    .merge(src_text,  on="source_chunk_id",     how="inner")
)


Source docs to evaluate: 20
Total pairs being sent to LLM: 96


,source_doc_id,embedding_score,suspicious_text,source_text
0,part13__source-document06022.txt,0.951554,"unable people, if it remains at peace. Commerc...",of soldiers. There is no art among a shepherd ...
1,part13__source-document06022.txt,0.903167,"are twice affairs every way yet separate, only...","to teach you yours. Nay, I knew that there oug..."
2,part13__source-document06022.txt,0.889145,"as i believed, of all painters whatsoever. And...","last of men to tell you so, had I trusted my o..."
3,part13__source-document06022.txt,0.851985,"and least, when removed some months from the e...","at the places where they exist, and cause a sl..."
4,part13__source-document06022.txt,0.849784,"and least, when removed some months from the e...","masses, which, at first, look quite definite; ..."


In [6]:
# ── Step 1: threshold filter ────────────────────────────────────────────────
import pandas as pd
LLM_SCORE_THRESHOLD = 0.95

llm_scores_df = pd.read_parquet("../05_text_alignment/llm_scores_df.parquet")
confirmed_sources = llm_scores_df[llm_scores_df["llm_score"] >= LLM_SCORE_THRESHOLD].copy()
print(f"Confirmed source docs (score >= {LLM_SCORE_THRESHOLD}): {len(confirmed_sources)}")
confirmed_sources[["source_doc_id", "llm_score", "llm_is_likely_source"]]


Confirmed source docs (score >= 0.95): 3


,source_doc_id,llm_score,llm_is_likely_source
0,part10__source-document04659.txt,0.95,True
1,part13__source-document06022.txt,0.95,True
2,part23__source-document11043.txt,0.95,True


In [7]:
# ── Step 2: classify plagiarism type per chunk pair ─────────────────────────
import time, json, re
from ollama import chat
from tqdm import tqdm

OLLAMA_MODEL = "gemma4:e4b"

PLAGIARISM_TYPES = ["copy_paste", "paraphrase", "shake", "none"]

def classify_chunk_pair(suspicious_text: str, source_text: str) -> dict:
    prompt = (
        "You are a plagiarism detection expert.\n\n"
        "Compare the SUSPICIOUS chunk and the SOURCE chunk below.\n\n"
        f"SUSPICIOUS:\n{suspicious_text}\n\n"
        f"SOURCE:\n{source_text}\n\n"
        "Classify the relationship into exactly one of these types:\n"
        "  - copy_paste : text is copied verbatim or near-verbatim (< 5% change)\n"
        "  - paraphrase : meaning preserved but sentences restructured or rewritten\n"
        "  - shake      : words replaced by synonyms / light edits, same structure\n"
        "  - none       : no meaningful plagiarism detected\n\n"
        "Also rate your confidence (0.0-1.0).\n"
        "Respond with ONLY a JSON object — no markdown — with keys: "
        "plagiarism_type (string), confidence (float 0-1), reasoning (string)."
    )

    response = chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
        think=False,
    )

    raw = response.message.content
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON in response: {raw[:300]}")
    data = json.loads(match.group())

    ptype = data.get("plagiarism_type", "none")
    if ptype not in PLAGIARISM_TYPES:
        ptype = "none"

    return {
        "plagiarism_type": ptype,
        "type_confidence": float(data.get("confidence", 0.0)),
        "type_reasoning": data.get("reasoning", ""),
    }


# Only run on pairs whose source doc passed the threshold
confirmed_source_ids = set(confirmed_sources["source_doc_id"].tolist())

confirmed_pairs = pairs_df[
    pairs_df["source_doc_id"].isin(confirmed_source_ids)
].copy()

print(f"Chunk pairs to classify: {len(confirmed_pairs)}")


Chunk pairs to classify: 555


In [7]:
# ── Step 3: run classification ───────────────────────────────────────────────
classification_rows = []

for row in tqdm(confirmed_pairs.itertuples(index=False), total=len(confirmed_pairs), desc="Classifying pairs"):
    try:
        result = classify_chunk_pair(row.suspicious_text, row.source_text)
    except Exception as e:
        result = {"plagiarism_type": "none", "type_confidence": 0.0, "type_reasoning": f"Error: {e}"}

    classification_rows.append({
        "suspicious_chunk_id":  row.suspicious_chunk_id,
        "suspicious_doc_id":    row.suspicious_doc_id,
        "suspicious_start_char": row.suspicious_start_char,
        "suspicious_end_char":  row.suspicious_end_char,
        "source_chunk_id":      row.source_chunk_id,
        "source_doc_id":        row.source_doc_id,
        "source_start_char":    row.source_start_char,
        "source_end_char":      row.source_end_char,
        "embedding_score":      row.embedding_score,
        "suspicious_text":      row.suspicious_text,
        "source_text":          row.source_text,
        **result,
    })

alignment_df = pd.DataFrame(classification_rows)
alignment_df = alignment_df.sort_values(
    ["source_doc_id", "embedding_score"], ascending=[True, False]
).reset_index(drop=True)

print(f"\nClassification summary:")
print(alignment_df["plagiarism_type"].value_counts().to_string())
alignment_df[["source_doc_id", "suspicious_start_char", "source_start_char", "embedding_score", "plagiarism_type", "type_confidence"]].head(10)


Classifying pairs:   0%|          | 0/555 [00:00<?, ?it/s]

Classifying pairs:   5%|▍         | 27/555 [01:11<23:09,  2.63s/it]


KeyboardInterrupt: 

In [ ]:
# ── Step 4: save span-level results (ready for PAN 2011 analytics) ──────────
OUTPUT_PATH = PROCESSED_DIR / "alignment_classified_spans.parquet"

alignment_df.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(alignment_df)} classified spans to: {OUTPUT_PATH}")

# Quick breakdown per source doc + plagiarism type
summary = (
    alignment_df[alignment_df["plagiarism_type"] != "none"]
    .groupby(["source_doc_id", "plagiarism_type"])
    .agg(count=("plagiarism_type", "size"), mean_confidence=("type_confidence", "mean"))
    .reset_index()
    .sort_values(["mean_confidence", "count"], ascending=[False, False])
)
summary


Saved 212 classified spans to: ..\..\..\datasets\processed\PAN2011_300\alignment_classified_spans.parquet


,source_doc_id,plagiarism_type,count,mean_confidence
1,part13__source-document06022.txt,copy_paste,4,0.950000
3,part13__source-document06022.txt,shake,10,0.910000
2,part13__source-document06022.txt,paraphrase,44,0.887500
0,part10__source-document04659.txt,paraphrase,4,0.875000
4,part23__source-document11043.txt,paraphrase,7,0.871429


In [ ]:
# ── Step 5a: keep best non-none span per suspicious chunk ────────────────────
best_spans = (
    alignment_df
    .assign(is_plagiarism=(alignment_df["plagiarism_type"] != "none").astype(int))
    .sort_values(["is_plagiarism", "type_confidence"], ascending=[False, False])
    .drop_duplicates(subset=["suspicious_chunk_id"], keep="first")
    .drop(columns=["is_plagiarism"])
    .reset_index(drop=True)
)

detected_chunks = (
    best_spans[best_spans["plagiarism_type"] != "none"]
    .sort_values(["source_doc_id", "suspicious_start_char"])
    .reset_index(drop=True)
    .copy()
)

print(f"Total classified rows:     {len(alignment_df)}")
print(f"After dedup (1 per chunk): {len(best_spans)}")
print(f"Detected chunk-level:      {len(detected_chunks)}")

# ── Step 5b: merge consecutive chunks into contiguous plagiarism spans ────────
# Merges chunks from the same source doc whose suspicious ranges are adjacent
# (gap ≤ MAX_GAP).  Keeps the highest-priority plagiarism type in each merged span.
MAX_GAP   = 1800  # ~1 chunk width — absorbs sliding-window overlaps
TYPE_RANK = {"copy_paste": 3, "shake": 2, "paraphrase": 1, "none": 0}

merged_rows = []
for _, grp in detected_chunks.groupby("source_doc_id"):
    grp = grp.sort_values("suspicious_start_char").reset_index(drop=True)
    current = grp.iloc[0].to_dict()

    for _, row in grp.iloc[1:].iterrows():
        gap = row["suspicious_start_char"] - current["suspicious_end_char"]
        if gap <= MAX_GAP:
            current["suspicious_end_char"] = max(current["suspicious_end_char"], row["suspicious_end_char"])
            current["source_start_char"]   = min(current["source_start_char"],   row["source_start_char"])
            current["source_end_char"]     = max(current["source_end_char"],     row["source_end_char"])
            if TYPE_RANK.get(row["plagiarism_type"], 0) > TYPE_RANK.get(current["plagiarism_type"], 0):
                current["plagiarism_type"] = row["plagiarism_type"]
                current["type_confidence"] = row["type_confidence"]
                current["type_reasoning"]  = row["type_reasoning"]
        else:
            merged_rows.append(current)
            current = row.to_dict()

    merged_rows.append(current)

detected = pd.DataFrame(merged_rows).reset_index(drop=True)

print(f"After merging adjacent:    {len(detected)} contiguous spans")
print()
detected[["source_doc_id", "suspicious_start_char", "suspicious_end_char",
          "plagiarism_type", "type_confidence"]]

In [ ]:
# ── Step 6: evaluate merged spans vs XML ground truth ────────────────────────
# A detected span is a TRUE POSITIVE if:
#   (1) source_doc_id matches the GT source_doc_id, AND
#   (2) the suspicious char range overlaps the GT suspicious range
# Source-side char offsets are NOT required to match because source docs are
# large concatenated files; semantically similar chunks may land in different
# sections.  Suspicious-side overlap is the authoritative criterion.

GROUND_TRUTH_PATH = Path("../../../datasets/processed/PAN2011_ground_truth/pan2011_plagiarism_spans.parquet")

gt_df  = pd.read_parquet(GROUND_TRUTH_PATH)
gt_doc = gt_df[gt_df["suspicious_doc_id"] == SUSPICIOUS_DOC_ID].copy()

def overlaps(a_start, a_end, b_start, b_end):
    return not (a_end <= b_start or a_start >= b_end)

hit_flags         = []
matched_gt_indices = set()

for _, det in detected.iterrows():
    hit = False
    for gt_idx, gt in gt_doc.iterrows():
        if det["source_doc_id"] != gt["source_doc_id"]:
            continue
        if overlaps(det["suspicious_start_char"], det["suspicious_end_char"],
                    gt["suspicious_offset"],      gt["suspicious_end"]):
            hit = True
            matched_gt_indices.add(gt_idx)
    hit_flags.append(hit)

detected = detected.copy()
detected["matched_gt"] = hit_flags

hits      = detected["matched_gt"].sum()
missed_gt = len(gt_doc) - len(matched_gt_indices)

precision = hits / len(detected)               if len(detected) else 0
recall    = len(matched_gt_indices) / len(gt_doc) if len(gt_doc)  else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print(f"XML ground truth spans:  {len(gt_doc)}")
print(f"Merged detected spans:   {len(detected)}")
print()
print("=== Detection vs Ground Truth ===")
print(f"GT spans matched:        {len(matched_gt_indices)}")
print(f"GT spans missed:         {missed_gt}")
print(f"True  positives:         {hits}")
print(f"False positives:         {len(detected) - hits}")
print()
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1:        {f1:.3f}")
print()

# Span size comparison: merged detected vs GT
detected["det_susp_len"] = detected["suspicious_end_char"] - detected["suspicious_start_char"]
gt_doc["gt_susp_len"]    = gt_doc["suspicious_end"]        - gt_doc["suspicious_offset"]

print("=== Span size: detected vs GT (suspicious side, chars) ===")
print(f"Detected — min={detected['det_susp_len'].min():.0f}  "
      f"median={detected['det_susp_len'].median():.0f}  "
      f"max={detected['det_susp_len'].max():.0f}")
print(f"GT       — min={gt_doc['gt_susp_len'].min():.0f}  "
      f"median={gt_doc['gt_susp_len'].median():.0f}  "
      f"max={gt_doc['gt_susp_len'].max():.0f}")
print()

# Per-type hit breakdown
print("=== Hit rate by plagiarism type ===")
print(detected.groupby("plagiarism_type")["matched_gt"].value_counts().unstack(fill_value=0).to_string())